# Hiệu chỉnh SASRec bằng học tương phản theo độ phổ biến

So sánh SASRec gốc, xóa item đồng đều và xóa item theo độ phổ biến trên Diginetica.

## 1. Chuẩn bị môi trường

### 1.1. Nạp mã nguồn

Nạp mã nguồn và cấu hình dùng cho thí nghiệm.

In [ ]:
from pathlib import Path
import base64, hashlib, json, os, shutil, sys, zlib
_SOURCE_BUNDLE = '...'
PAYLOAD = json.loads(zlib.decompress(base64.b64decode(_SOURCE_BUNDLE)))
del _SOURCE_BUNDLE

### 1.2. Chuẩn bị dữ liệu

Xác định thư mục dự án và kiểm tra dữ liệu đầu vào.

In [ ]:
EXPECTED_DATA = {'item_mapping.csv': '7ef8636a4e74c9471f0e2d9805ae5b5c3fe3186dcb5e094f6fd2a86d75097f4a', 'train_examples.parquet': 'eb41c516b26ea163ce6d9cceefd6b6817d6c897f49dcc23d4ab0136d69225d04', 'validation_examples.parquet': '52cc7af596b5b708162781b9f24267bf49c4a0fd5e43f3d5c8df9302639e194c', 'test_examples.parquet': '54b5014dbfbb8a43507b06655421bca92930e8563aad41295cb6deaa37260668'}
ON_KAGGLE = Path('/kaggle/input').exists()
if ON_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working/contrastive_sasrec')
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    candidates = [p.parent for p in Path('/kaggle/input').rglob('item_mapping.csv') if (p.parent / 'train_examples.parquet').exists()]
    source = next((p for p in candidates if all(((p / name).exists() and hashlib.sha256((p / name).read_bytes()).hexdigest() == value for name, value in EXPECTED_DATA.items()))), None)
    if source is None:
        raise RuntimeError('Attach ngytfd/btl-k03-sasrec-data with the reviewed data split; hashes do not match.')
    target = PROJECT_ROOT / 'data/processed/diginetica'
    target.mkdir(parents=True, exist_ok=True)
    for name in EXPECTED_DATA:
        shutil.copy2(source / name, target / name)
    for relative, text in PAYLOAD.items():
        target_file = PROJECT_ROOT / relative
        target_file.parent.mkdir(parents=True, exist_ok=True)
        target_file.write_text(text, encoding='utf-8')
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'BTL-K03-tong-quan-de-tai.md').exists()))

### 1.3. Nạp thư viện

Nạp các thư viện và hàm huấn luyện.

In [ ]:
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
print('Project root:', PROJECT_ROOT)
import copy, json, subprocess, time
from datetime import datetime, timezone, timedelta
import pandas as pd
import torch
from btl_k03.utils import load_experiment_config, load_yaml
from btl_k03.contrastive import train_contrastive_sasrec, generate_popularity_aware_views, compute_infonce_contrastive_loss

## 2. Thiết lập thí nghiệm

### 2.1. Cấu hình huấn luyện

Thiết lập ba phương pháp, hai seed và tiêu chí chọn checkpoint.

In [ ]:
SMOKE_ONLY = not ON_KAGGLE
REQUIRE_GPU = ON_KAGGLE
STUDY = load_yaml(PROJECT_ROOT / 'configs/contrastive.yaml')

def study_config():
    cfg = load_experiment_config('sasrec')
    cfg['training'].update({k: v for k, v in STUDY['training'].items() if k != 'patience'})
    cfg['early_stopping'].update(patience=STUDY['training']['patience'], monitor='mrr@20')
    cfg['contrastive'] = copy.deepcopy(STUDY['contrastive'])
    return cfg

### 2.2. Thiết bị tính toán

Kiểm tra GPU và xác định thiết bị huấn luyện.

In [ ]:
if REQUIRE_GPU and (not torch.cuda.is_available()):
    raise RuntimeError('Enable a GPU accelerator before full training.')
GPU_COUNT = min(torch.cuda.device_count(), 2)
DEVICE = 'cuda:0' if GPU_COUNT else 'cpu'
torch.set_num_threads(2)

### 2.3. Thư mục kết quả

Tạo thư mục kết quả và lưu cấu hình thí nghiệm.

In [ ]:
run_id = datetime.now(timezone(timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
OUTPUT = PROJECT_ROOT / 'results/contrastive' / run_id
OUTPUT.mkdir(parents=True, exist_ok=False)
(OUTPUT / 'locked_study.json').write_text(json.dumps(STUDY, indent=2), encoding='utf-8')
print('Device:', DEVICE, '| GPUs:', GPU_COUNT)
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Methods:', STUDY['methods'], '| seeds:', STUDY['seeds'])
print('Output:', OUTPUT)

## 3. Kiểm tra trước huấn luyện

### 3.1. Phép biến đổi chuỗi

Kiểm tra thứ tự item và sự kiện cuối sau phép xóa.

In [ ]:
sequence = torch.tensor([[1, 2, 3, 4, 0], [1, 0, 0, 0, 0]], device=DEVICE)
head = torch.tensor([False, True, False, True, False], device=DEVICE)
v1, l1, v2, l2 = generate_popularity_aware_views(sequence, torch.tensor([4, 1], device=DEVICE), head, 1.0, 0.0)
assert v1.tolist() == [[2, 4, 0, 0, 0], [1, 0, 0, 0, 0]]
assert l1.tolist() == [2, 1]

### 3.2. Hàm mất mát tương phản

Kiểm tra giá trị loss và gradient khi các mẫu thuộc cùng session.

In [ ]:
z1 = torch.randn(4, 8, device=DEVICE, dtype=torch.float16, requires_grad=True)
z2 = torch.randn(4, 8, device=DEVICE, dtype=torch.float16, requires_grad=True)
loss = compute_infonce_contrastive_loss(z1, z2, session_ids=torch.ones(4, device=DEVICE, dtype=torch.long))
assert torch.isfinite(loss) and loss.item() == 0
loss.backward()
assert torch.isfinite(z1.grad).all()

### 3.3. Cấu hình chạy kiểm tra

Thiết lập một epoch trên tập dữ liệu nhỏ.

In [ ]:
smoke_cfg = study_config()
smoke_cfg['training'].update(epochs=1, train_batch_size=32, eval_batch_size=32)

### 3.4. Chạy kiểm tra ba phương pháp

Huấn luyện trên 64 mẫu train và đánh giá trên 64 mẫu validation.

In [ ]:
SMOKE_RESULTS = {}
for method in STUDY['methods']:
    value = train_contrastive_sasrec(2024, OUTPUT / 'smoke' / method, method=method, config=smoke_cfg, device_str=DEVICE, train_limit=64, validation_limit=64, evaluate_test=False)
    assert value['frozen_embedding_max_abs_drift'] == 0 and 'test' not in value
    SMOKE_RESULTS[method] = value
print('SMOKE PASSED:', list(SMOKE_RESULTS))
(OUTPUT / 'smoke_passed.json').write_text(json.dumps({m: r['best_validation_score'] for m, r in SMOKE_RESULTS.items()}, indent=2), encoding='utf-8')

## 4. Huấn luyện mô hình

### 4.1. Quy trình cho từng seed

Huấn luyện ba phương pháp và đánh giá test sau khi chọn checkpoint bằng validation.

In [ ]:
worker_code = "\nimport copy, sys\nfrom pathlib import Path\nimport torch\nfrom btl_k03.utils import load_experiment_config, load_yaml\nfrom btl_k03.contrastive import train_contrastive_sasrec\nseed, device, output = (int(sys.argv[1]), sys.argv[2], Path(sys.argv[3]))\ntorch.set_num_threads(2)\nstudy = load_yaml(Path('configs/contrastive.yaml'))\nfor method in study['methods']:\n    cfg = load_experiment_config('sasrec')\n    cfg['training'].update({k: v for k, v in study['training'].items() if k != 'patience'})\n    cfg['early_stopping'].update(patience=study['training']['patience'], monitor='mrr@20')\n    cfg['contrastive'] = copy.deepcopy(study['contrastive'])\n    train_contrastive_sasrec(seed, output / method / ('seed_' + str(seed)), method=method, config=cfg, device_str=device, evaluate_test=True)\n"

### 4.2. Phân bổ thiết bị

Phân bổ các seed cho tối đa hai GPU bằng các tiến trình riêng.

In [ ]:
import concurrent.futures
environment = dict(os.environ, PYTHONPATH=str(PROJECT_ROOT / 'src'), OMP_NUM_THREADS='2', MKL_NUM_THREADS='2')
slots = max(1, GPU_COUNT)

def worker(slot):
    for seed in STUDY['seeds'][slot::slots]:
        device = f'cuda:{slot}' if GPU_COUNT else 'cpu'
        print('START seed', seed, 'on', device, flush=True)
        subprocess.run([sys.executable, '-B', '-c', worker_code, str(seed), device, str(OUTPUT / 'full')], cwd=PROJECT_ROOT, env=environment, check=True)
        print('COMPLETE seed', seed, flush=True)

### 4.3. Chạy huấn luyện đầy đủ

Chạy thí nghiệm đầy đủ sau khi hoàn thành các bước kiểm tra.

In [ ]:
if SMOKE_ONLY:
    print('Chế độ kiểm tra: bỏ qua huấn luyện đầy đủ.')
else:
    with concurrent.futures.ThreadPoolExecutor(max_workers=slots) as pool:
        futures = [pool.submit(worker, slot) for slot in range(slots)]
        for future in futures:
            future.result()

## 5. Đánh giá kết quả

### 5.1. Bảng kết quả từng lần chạy

Tổng hợp Recall, MRR, độ phủ và Gini từ các kết quả đã lưu.

In [ ]:
if SMOKE_ONLY:
    summary_table = pd.DataFrame([{'method': m, 'status': 'smoke_passed', 'best_validation_score': r['best_validation_score'], 'cold_drift': r['frozen_embedding_max_abs_drift']} for m, r in SMOKE_RESULTS.items()])
else:
    rows = []
    for method in STUDY['methods']:
        for seed in STUDY['seeds']:
            result = json.loads((OUTPUT / 'full' / method / f'seed_{seed}' / 'metrics.json').read_text(encoding='utf-8'))
            accuracy = {r['target_group']: r for r in result['test']['ranking_by_group']}
            rows.append({'method': method, 'seed': seed, 'best_epoch': result['best_epoch'], 'recall@20': accuracy['all_warm']['recall@20'], 'mrr@20': accuracy['all_warm']['mrr@20'], 'rare_recall@20': accuracy['rare']['recall@20'], 'rare_mrr@20': accuracy['rare']['mrr@20'], 'coverage@20': result['test']['popularity_bias']['coverage_warm@20'], 'gini@20': result['test']['popularity_bias']['gini_warm@20'], 'cold_drift': result['frozen_embedding_max_abs_drift'], 'seconds': result['seconds']})
    summary_table = pd.DataFrame(rows)

### 5.2. Kết quả trung bình

Tính trung bình các chỉ số qua hai seed cho từng phương pháp.

In [ ]:
if not SMOKE_ONLY:
    metric_columns = [c for c in summary_table.columns if c not in ('method', 'seed')]
    method_means = summary_table.groupby('method')[metric_columns].mean()
    display(method_means)

### 5.3. So sánh bằng bootstrap

Ước lượng chênh lệch và khoảng tin cậy bằng lấy mẫu lại theo session.

In [ ]:
if not SMOKE_ONLY:
    from btl_k03.statistics import clustered_delta
    import numpy as np
    paired = []
    for seed in STUDY['seeds']:
        head = pd.read_csv(OUTPUT / 'full/head' / f'seed_{seed}' / 'test_per_query.csv.gz')
        for control in ('baseline', 'uniform'):
            base = pd.read_csv(OUTPUT / 'full' / control / f'seed_{seed}' / 'test_per_query.csv.gz')
            for group, metric in (('rare', 'hit'), ('all_warm', 'hit'), ('rare', 'reciprocal_rank')):
                paired.append({'seed': seed, 'control': control, 'group': group, 'metric': metric, **clustered_delta(base, head, metric, group, 2000, np.random.default_rng(seed))})

## 6. Lưu kết quả

Lưu bảng so sánh, kết quả trung bình và khoảng tin cậy.

In [ ]:
summary_table.to_csv(OUTPUT / 'comparison.csv', index=False)
if not SMOKE_ONLY:
    method_means.to_csv(OUTPUT / 'method_means.csv')
    pd.DataFrame(paired).to_csv(OUTPUT / 'paired_session_bootstrap.csv', index=False)
display(summary_table)
print('Thư mục kết quả:', OUTPUT)